In [12]:
# Cell 1 — imports + config
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Config:
    dim = 32
    num_experts = 4
    hidden = 64
    batch_size = 256
    steps = 1500
    lr = 1e-3
    lb_weight = 0.1
    aux_weight = 0.5
    top_k = 1

cfg = Config()
print(cfg.__dict__)

{}


In [13]:
# Cell 2 — clustered Gaussian mixture dataset

torch.manual_seed(0)

cluster_means = torch.tensor([
    +3.0,   # Expert 0 cluster center
    -3.0,   # Expert 1 cluster center
    +6.0,   # Expert 2 cluster center
    -6.0    # Expert 3 cluster center
]).view(cfg.num_experts, 1)

def make_batch(batch_size):
    expert_ids = torch.randint(0, cfg.num_experts, (batch_size,))
    x = torch.randn(batch_size, cfg.dim)

    # shift each sample toward its cluster mean
    for i in range(batch_size):
        eid = int(expert_ids[i])
        x[i] += cluster_means[eid]

    return x.to(device), expert_ids.to(device)

In [14]:
# Cell 3 — expert-specific transforms

W0 = torch.randn(cfg.dim, cfg.dim)
W1 = torch.randn(cfg.dim, cfg.dim)
W2 = torch.randn(cfg.dim, cfg.dim)
W3 = torch.randn(cfg.dim, cfg.dim)

def expert_map(x, eid):
    if eid == 0:
        return x @ W0
    elif eid == 1:
        return torch.relu(x @ W1)
    elif eid == 2:
        return torch.sin(x @ W2)
    elif eid == 3:
        return torch.tanh(x @ W3)

In [17]:
# Cell 3.5 — move expert matrices to the correct device

W0 = W0.to(device)
W1 = W1.to(device)
W2 = W2.to(device)
W3 = W3.to(device)

In [18]:
# Cell 4 — MoE model

class Expert(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, dim)
        )

    def forward(self, x):
        return self.net(x)

class MoE(nn.Module):
    def __init__(self, dim, num_experts, hidden):
        super().__init__()
        self.num_experts = num_experts

        # deep router
        self.router = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_experts)
        )

        self.experts = nn.ModuleList([Expert(dim, hidden) for _ in range(num_experts)])

    def forward(self, x):
        B = x.size(0)

        logits = self.router(x)
        probs = F.softmax(logits, dim=-1)

        top1 = torch.argmax(probs, dim=-1)
        one_hot = F.one_hot(top1, num_classes=self.num_experts).float()

        outputs = torch.zeros_like(x)
        for e in range(self.num_experts):
            mask = one_hot[:, e].bool()
            if mask.any():
                outputs[mask] = self.experts[e](x[mask])

        frac_tokens = one_hot.mean(dim=0)
        mean_gate = probs.mean(dim=0)
        lb_loss = self.num_experts * torch.sum(frac_tokens * mean_gate)

        stats = {
            "lb_loss": lb_loss,
            "entropy": (-probs * probs.clamp(min=1e-8).log()).sum(dim=-1).mean().item(),
            "ia": probs.max(dim=-1).values.mean().item()
        }

        return outputs, stats, logits

In [19]:
# Cell 5 — training loop

moe = MoE(cfg.dim, cfg.num_experts, cfg.hidden).to(device)
opt = torch.optim.Adam(moe.parameters(), lr=cfg.lr)

def mse_loss(pred, target):
    return ((pred - target) ** 2).mean()

for step in range(cfg.steps + 1):
    x, expert_ids = make_batch(cfg.batch_size)

    # compute target outputs using expert_map
    y = torch.stack([expert_map(x[i], int(expert_ids[i])) for i in range(cfg.batch_size)], dim=0)

    pred, stats, logits = moe(x)

    loss_main = mse_loss(pred, y)
    loss_lb = stats["lb_loss"]
    loss_route = F.cross_entropy(logits, expert_ids)

    loss = loss_main + cfg.lb_weight * loss_lb + cfg.aux_weight * loss_route

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 100 == 0:
        print(
            f"Step {step} | "
            f"Loss {loss_main.item():.4f} | "
            f"Ent {stats['entropy']:.4f} | "
            f"LB {loss_lb.item():.4f} | "
            f"IA {stats['ia']:.4f} | "
            f"Route {loss_route.item():.4f}"
        )

Step 0 | Loss 104.1248 | Ent 1.0502 | LB 1.2188 | IA 0.5482 | Route 0.9396
Step 100 | Loss 47.3996 | Ent 0.6409 | LB 1.0996 | IA 0.6430 | Route 0.4969
Step 200 | Loss 29.8629 | Ent 0.5688 | LB 1.0237 | IA 0.7134 | Route 0.3608
Step 300 | Loss 9.8372 | Ent 0.4551 | LB 1.0129 | IA 0.8180 | Route 0.2064
Step 400 | Loss 7.9506 | Ent 0.3338 | LB 1.0045 | IA 0.8892 | Route 0.1197
Step 500 | Loss 6.6554 | Ent 0.2414 | LB 1.0077 | IA 0.9295 | Route 0.0744
Step 600 | Loss 5.9786 | Ent 0.1634 | LB 1.0035 | IA 0.9590 | Route 0.0424
Step 700 | Loss 4.6678 | Ent 0.1411 | LB 1.0167 | IA 0.9657 | Route 0.0352
Step 800 | Loss 3.4439 | Ent 0.1042 | LB 1.0025 | IA 0.9767 | Route 0.0237
Step 900 | Loss 3.0018 | Ent 0.0862 | LB 1.0096 | IA 0.9816 | Route 0.0187
Step 1000 | Loss 2.1577 | Ent 0.0640 | LB 1.0078 | IA 0.9874 | Route 0.0128
Step 1100 | Loss 1.7515 | Ent 0.0547 | LB 1.0161 | IA 0.9895 | Route 0.0106
Step 1200 | Loss 1.2752 | Ent 0.0412 | LB 1.0115 | IA 0.9926 | Route 0.0075
Step 1300 | Loss 0.9